# DDPM — Chequeo de métricas físicas (Kaggle / T4)

**Objetivo**: vistazo rápido para verificar que las métricas físicas de `metrics.py` funcionan bien sobre imágenes generadas por el **DDPM** (único modelo en este cuadernillo).

**Protocolo estadístico**:
- Se muestrean `N` conjuntos de parámetros θ del **split de test** (SEED=42, 70/15/15), estratificados por **fase magnética** (6 fases, proporcional a su frecuencia en el test).
- Por cada θ se generan `K` imágenes desde el DDPM (muestreo estocástico del ruido inicial).
- Cada métrica se calcula **por muestra** (gen_i vs original) y luego se promedia sobre las K — nunca se promedian las imágenes.
- Resultados **generales** y **por fase**.

**Config inicial**: `N=2000`, `K=4`. Calibrado para correr en ~20–35 min en una T4.

**Inputs (Kaggle datasets)**:
- `carloscanamejoy/dataset-spines-united-v2` → `dataset_unificado_v2.npz`
- `carloscanamejoy/weights-models` → `ddpm_spines_final_39crop.pt`
- `carloscanamejoy/physicalmetrics` → `metrics.py`

In [ ]:
# ============================================================
# CONFIGURACIÓN — edita aquí
# ============================================================
N = 2000          # nº de conjuntos de parámetros θ a evaluar (muestreados del test)
K = 4             # nº de imágenes generadas por θ (para promediar métricas)
SEED = 42         # semilla del split (debe coincidir con el entrenamiento del DDPM)
SAMPLE_SEED = 123 # semilla del muestreo estratificado y del ruido de generación
FAST_STEPS = 100  # pasos del sampler DDIM rápido
GEN_BATCH = 64    # batch de generación (imágenes simultáneas en GPU)

print(f"N={N} θ  ×  K={K} muestras  =  {N*K:,} generaciones | DDIM {FAST_STEPS} pasos")

In [ ]:
# ============================================================
# Setup: imports y localización de archivos (Kaggle o Colab)
# ============================================================
import os, sys, time, math, importlib.util
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def _find(candidates, what):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No encuentro {what}. Probé: {candidates}")

DATASET_PATH = _find([
    "/kaggle/input/dataset-spines-united-v2/dataset_unificado_v2.npz",
    "/content/data/dataset_unificado_v2.npz",
], "dataset_unificado_v2.npz")

DDPM_CKPT = _find([
    "/kaggle/input/weights-models/ddpm_spines_final_39crop.pt",
    "/content/weights/ddpm_spines_final_39crop.pt",
], "ddpm_spines_final_39crop.pt")

METRICS_PATH = _find([
    "/kaggle/input/physicalmetrics/metrics.py",
    "/content/weights/metrics.py",
    os.path.join("..", "utils", "metrics.py"),
], "metrics.py")

print(f"dataset: {DATASET_PATH}")
print(f"ddpm:    {DDPM_CKPT}")
print(f"metrics: {METRICS_PATH}")

In [ ]:
# ============================================================
# Cargar metrics.py compartido
# ============================================================
spec = importlib.util.spec_from_file_location("metrics", METRICS_PATH)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules["metrics"] = metrics

from metrics import (
    STRUCTURE_MAP, STRUCTURE_NAMES, STRUCTURE_COLORS, MASK,
    get_structure_label,
    masked_ssim, masked_mse,
    magnetization, abs_magnetization, cnn_correlation,
    structure_factor, azimuthal_average, oz_fit,
    center_crop, apply_figure_style,
)
apply_figure_style()
print("metrics.py cargado.")
print(f"Fases ({len(STRUCTURE_NAMES)}): {STRUCTURE_NAMES}")

In [ ]:
# ============================================================
# Cargar dataset y reconstruir el split de test (SEED=42, 100%)
# Réplica EXACTA de make_split() del notebook de entrenamiento del DDPM:
#   - subsample 100% con rng.choice(SEED)
#   - split 70/15/15 con train_test_split(random_state=SEED)
#   - MinMaxScaler ajustado SOLO en train
# Esto reproduce el mismo test set y el mismo scaler que vio el DDPM.
# ============================================================
data   = np.load(DATASET_PATH)
imgs   = data["img"].astype(np.float32)
params = data["params"].astype(np.float32)
labels = data["labels"].astype(int)
if imgs.ndim == 3:
    imgs = imgs[..., np.newaxis]
Nfull = len(imgs)
print(f"dataset: {imgs.shape}  params {params.shape}  | clusters {sorted(np.unique(labels).tolist())}")

rng = np.random.RandomState(SEED)
sub_idx = rng.choice(Nfull, size=int(Nfull * 1.0), replace=False)   # 100%

idx_all = np.arange(len(sub_idx))
idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.30, random_state=SEED)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.50, random_state=SEED)

scaler = MinMaxScaler().fit(params[sub_idx][idx_tr])

# Índices globales del test set + sus etiquetas y fase
test_global = sub_idx[idx_te]
test_imgs   = imgs[test_global]                         # 39×39 (originales)
test_params = params[test_global]
test_phase  = np.array([get_structure_label(c) for c in labels[test_global]])
print(f"test set: {len(test_global):,} muestras")

# Distribución de fases en el test
print("\nDistribución de fases en el test:")
for ph in STRUCTURE_NAMES:
    cnt = int((test_phase == ph).sum())
    print(f"  {ph:24s} {cnt:6,}  ({100*cnt/len(test_phase):5.1f}%)")

In [ ]:
# ============================================================
# Muestreo estratificado por fase: cuota proporcional a la
# frecuencia de cada fase en el test. Ej.: si Helical es 32%
# del test y N=2000 → ~640 muestras Helical.
# ============================================================
samp_rng = np.random.RandomState(SAMPLE_SEED)
sel_local = []
for ph in STRUCTURE_NAMES:
    ph_idx = np.where(test_phase == ph)[0]
    if len(ph_idx) == 0:
        continue
    quota = int(round(N * len(ph_idx) / len(test_phase)))
    quota = min(quota, len(ph_idx))                      # no exceder lo disponible
    if quota > 0:
        sel_local.append(samp_rng.choice(ph_idx, size=quota, replace=False))
sel_local = np.concatenate(sel_local)
samp_rng.shuffle(sel_local)

eval_imgs   = test_imgs[sel_local]                       # (N, 39, 39, 1)
eval_params = test_params[sel_local]                     # (N, 8) físicos
eval_phase  = test_phase[sel_local]                      # (N,)
eval_cond   = scaler.transform(eval_params).astype(np.float32)  # θ normalizado para el DDPM

print(f"Seleccionadas {len(sel_local):,} θ (objetivo N={N}). Reparto real por fase:")
for ph in STRUCTURE_NAMES:
    print(f"  {ph:24s} {int((eval_phase==ph).sum()):5,}")

In [ ]:
# ============================================================
# Arquitectura del DDPM (idéntica a ddpm_spines_train.ipynb)
# ============================================================
IMG_SIZE = 40
CROP_TO  = 39
COND_DIM = 8
T_STEPS  = 1000
BETA_START, BETA_END = 1e-4, 0.02

class DDPMScheduler:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule='cosine', device='cpu'):
        self.T = T
        if schedule == 'linear':
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        elif schedule == 'cosine':
            steps = T + 1; s = 0.008
            x = torch.linspace(0, T, steps, device=device)
            ac = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
            ac = ac / ac[0]
            betas = (1 - (ac[1:] / ac[:-1])).clamp(max=0.999)
        else:
            raise ValueError(schedule)
        alphas = 1.0 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.sqrt_alphas_cumprod = ac.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0 - ac).sqrt()
        self.betas = betas; self.alphas = alphas; self.alphas_cumprod = ac

def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1))
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)

class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
    def forward(self, t, cond):
        t_emb = sinusoidal_embedding(t, self.t_mlp[0].in_features)
        return self.t_mlp(t_emb) + self.c_mlp(cond)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch); self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch); self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, emb):
        h = F.silu(self.norm1(x)); h = self.conv1(h)
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = F.silu(self.norm2(h)); h = self.dropout(h); h = self.conv2(h)
        return h + self.skip(x)

class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch); self.qkv = nn.Conv2d(ch, ch*3, 1); self.proj = nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x); q, k, v = self.qkv(h).chunk(3, dim=1)
        q = q.reshape(B, C, -1); k = k.reshape(B, C, -1); v = v.reshape(B, C, -1)
        attn = torch.softmax(torch.bmm(q.transpose(1, 2), k) / math.sqrt(C), dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).reshape(B, C, H, W)
        return x + self.proj(out)

class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1, 2, 4), cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        chs = [base_ch * m for m in ch_mults]
        self.emb = TimeCondEmbedding(t_dim=emb_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in = nn.Conv2d(img_channels, chs[0], 3, padding=1)
        self.down_blocks = nn.ModuleList(); self.down_samples = nn.ModuleList()
        in_ch = chs[0]; self.skip_channels = []
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout)]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs)-1 else nn.Identity())
            in_ch = out_ch
        self.mid_block1 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.mid_attn = SelfAttention(chs[-1])
        self.mid_block2 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.up_blocks = nn.ModuleList(); self.up_samples = nn.ModuleList()
        for i, out_ch in enumerate(reversed(chs)):
            skip_ch = self.skip_channels[-(i + 1)]
            self.up_blocks.append(nn.ModuleList([
                ResBlock(in_ch + skip_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout)]))
            self.up_samples.append(nn.ConvTranspose2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs)-1 else nn.Identity())
            in_ch = out_ch
        self.norm_out = nn.GroupNorm(8, chs[0]); self.conv_out = nn.Conv2d(chs[0], img_channels, 1)
    def forward(self, x, t, cond):
        emb = self.emb(t, cond); h = self.conv_in(x); skips = []
        for (rb1, rb2), ds in zip(self.down_blocks, self.down_samples):
            h = rb1(h, emb); h = rb2(h, emb); skips.append(h); h = ds(h)
        h = self.mid_block1(h, emb); h = self.mid_attn(h); h = self.mid_block2(h, emb)
        for (rb1, rb2), us, skip in zip(self.up_blocks, self.up_samples, reversed(skips)):
            h = torch.cat([h, skip], dim=1); h = rb1(h, emb); h = rb2(h, emb); h = us(h)
        h = F.silu(self.norm_out(h))
        return self.conv_out(h)

@torch.no_grad()
def fast_sample(model, cond, scheduler, n_steps=100, img_size=40):
    """Sampler DDIM determinista (idéntico al de entrenamiento)."""
    B = cond.shape[0]
    x = torch.randn(B, 1, img_size, img_size, device=cond.device)
    timesteps = list(range(0, scheduler.T, scheduler.T // n_steps))[::-1]
    for t_val in timesteps:
        t_tensor = torch.full((B,), t_val, device=cond.device, dtype=torch.long)
        eps_pred = model(x, t_tensor, cond)
        sqrt_a = scheduler.sqrt_alphas_cumprod[t_val]
        sqrt_1a = scheduler.sqrt_one_minus_alphas_cumprod[t_val]
        x0_pred = ((x - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)
        if t_val > 0:
            t_prev = max(t_val - scheduler.T // n_steps, 0)
            sqrt_a_prev = scheduler.sqrt_alphas_cumprod[t_prev]
            sqrt_1a_prev = scheduler.sqrt_one_minus_alphas_cumprod[t_prev]
            x = sqrt_a_prev * x0_pred + sqrt_1a_prev * eps_pred
        else:
            x = x0_pred
    return x

print("Arquitectura DDPM definida.")

In [ ]:
# ============================================================
# Cargar checkpoint DDPM (usa pesos EMA si existen)
# ============================================================
ckpt = torch.load(DDPM_CKPT, map_location=DEVICE, weights_only=False)
hp = ckpt.get("hyperparams", {"base_ch": 80, "cond_emb_dim": 128, "dropout": 0.1, "beta_schedule": "cosine"})

model = ConditionalUNet(
    img_channels=1, base_ch=hp["base_ch"], ch_mults=(1, 2, 4),
    cond_dim=COND_DIM, emb_dim=hp["cond_emb_dim"], dropout=hp["dropout"],
).to(DEVICE)

model.load_state_dict(ckpt["model"])
if ckpt.get("ema") is not None:
    print("Aplicando pesos EMA.")
    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.requires_grad and n in ckpt["ema"]:
                p.data.copy_(ckpt["ema"][n].to(DEVICE))
model.eval()

scheduler = DDPMScheduler(T=T_STEPS, beta_start=BETA_START, beta_end=BETA_END,
                          schedule=hp["beta_schedule"], device=DEVICE)
print(f"DDPM cargado: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params | schedule={hp['beta_schedule']}")

In [ ]:
# ============================================================
# Métricas físicas por imagen
# Compara una imagen generada (39×39) contra su original (39×39).
# Para escalares físicos reportamos el |error| respecto al original;
# para SSIM/MSE la similitud directa. Todas las imágenes en [-1,1].
# ============================================================
def physical_metrics(gen39, real39):
    """Devuelve un dict de métricas para una pareja (gen, real) de 39×39."""
    out = {}
    # --- fidelidad de imagen ---
    out["ssim"] = float(masked_ssim(real39, gen39))
    out["mse"]  = float(masked_mse(real39, gen39))
    # --- magnetización (escalar físico) ---
    out["mag_gen"]  = float(magnetization(gen39))
    out["mag_real"] = float(magnetization(real39))
    out["mag_abs_err"] = abs(out["mag_gen"] - out["mag_real"])
    # --- correlación de vecinos cercanos ---
    out["corr_gen"]  = float(cnn_correlation(gen39))
    out["corr_real"] = float(cnn_correlation(real39))
    out["corr_abs_err"] = abs(out["corr_gen"] - out["corr_real"])
    # --- pico del factor de estructura S(q): posición del máximo radial ---
    q_g, sq_g = azimuthal_average(structure_factor(gen39))
    q_r, sq_r = azimuthal_average(structure_factor(real39))
    out["qpeak_gen"]  = float(q_g[np.argmax(sq_g)]) if len(sq_g) else np.nan
    out["qpeak_real"] = float(q_r[np.argmax(sq_r)]) if len(sq_r) else np.nan
    out["qpeak_abs_err"] = abs(out["qpeak_gen"] - out["qpeak_real"])
    # --- Ornstein-Zernike: longitud de correlación ξ (solo si el fit es válido) ---
    chi_g, xi_g, r2_g = oz_fit(gen39)
    out["xi_gen"] = float(xi_g); out["oz_r2_gen"] = float(r2_g)
    return out

METRIC_KEYS = ["ssim", "mse", "mag_abs_err", "corr_abs_err", "qpeak_abs_err", "xi_gen", "oz_r2_gen"]
print("Métricas por imagen:", METRIC_KEYS)

In [ ]:
# ============================================================
# Loop principal: generar K muestras por θ y calcular métricas
# por muestra. Se guarda la media sobre las K por cada θ.
# ============================================================
torch.manual_seed(SAMPLE_SEED)
n_theta = len(eval_cond)

# Acumuladores: para cada θ, la media de sus K muestras por métrica.
per_theta = {k: np.full(n_theta, np.nan, dtype=np.float64) for k in METRIC_KEYS}

t0 = time.time()
for i0 in range(0, n_theta, GEN_BATCH):
    i1 = min(i0 + GEN_BATCH, n_theta)
    cond_batch = torch.from_numpy(eval_cond[i0:i1]).to(DEVICE)
    reals = eval_imgs[i0:i1, :, :, 0]                     # (b, 39, 39)
    b = i1 - i0
    # acumular métricas por muestra para promediar sobre K
    acc = {k: np.zeros((b, K), dtype=np.float64) for k in METRIC_KEYS}
    for kk in range(K):
        gen40 = fast_sample(model, cond_batch, scheduler, n_steps=FAST_STEPS, img_size=IMG_SIZE)
        gen39 = gen40[:, 0, :CROP_TO, :CROP_TO].cpu().numpy()   # crop 40→39
        for j in range(b):
            m = physical_metrics(gen39[j], reals[j])
            for key in METRIC_KEYS:
                acc[key][j, kk] = m[key]
    for key in METRIC_KEYS:
        per_theta[key][i0:i1] = np.nanmean(acc[key], axis=1)   # media sobre K
    if (i0 // GEN_BATCH) % 5 == 0:
        done = i1
        rate = done / (time.time() - t0 + 1e-9)
        eta = (n_theta - done) / rate / 60
        print(f"  {done:>5}/{n_theta} θ | {rate:.1f} θ/s | ETA {eta:.1f} min")

elapsed = (time.time() - t0) / 60
print(f"\nGeneración + métricas: {elapsed:.1f} min  ({n_theta*K:,} imágenes)")

In [ ]:
# ============================================================
# Resultados GENERALES (media ± std sobre los θ del test)
# ============================================================
def summarize(mask=None):
    sel = np.ones(n_theta, dtype=bool) if mask is None else mask
    rows = {}
    for key in METRIC_KEYS:
        vals = per_theta[key][sel]
        vals = vals[~np.isnan(vals)]
        rows[key] = (float(np.mean(vals)) if len(vals) else np.nan,
                     float(np.std(vals))  if len(vals) else np.nan,
                     len(vals))
    return rows

gen_summary = summarize()
print("="*64)
print(f"RESULTADOS GENERALES — DDPM ({n_theta:,} θ × {K} muestras)")
print("="*64)
print(f"{'métrica':>16}  {'media':>10}  {'std':>10}  {'n':>6}")
for key, (mu, sd, n_) in gen_summary.items():
    print(f"{key:>16}  {mu:>10.4f}  {sd:>10.4f}  {n_:>6}")

In [ ]:
# ============================================================
# Resultados POR FASE (6 estructuras magnéticas)
# ============================================================
import pandas as pd
phase_rows = []
for ph in STRUCTURE_NAMES:
    mask = (eval_phase == ph)
    if mask.sum() == 0:
        continue
    s = summarize(mask)
    row = {"fase": ph, "n_θ": int(mask.sum())}
    for key in METRIC_KEYS:
        row[key] = s[key][0]
    phase_rows.append(row)
df_phase = pd.DataFrame(phase_rows).set_index("fase")
print("RESULTADOS POR FASE (media sobre θ de cada fase):")
df_phase.round(4)

In [ ]:
# ============================================================
# Gráficas: métricas por fase (barras) — un panel por métrica
# ============================================================
plot_keys = ["ssim", "mse", "mag_abs_err", "corr_abs_err", "qpeak_abs_err", "oz_r2_gen"]
titles = {
    "ssim": "SSIM (↑ mejor)", "mse": "MSE (↓ mejor)",
    "mag_abs_err": "|Δ magnetización| (↓)", "corr_abs_err": "|Δ correlación NN| (↓)",
    "qpeak_abs_err": "|Δ q_pico S(q)| (↓)", "oz_r2_gen": "R² del fit OZ (gen)",
}
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
phases = list(df_phase.index)
colors = [STRUCTURE_COLORS.get(p, "#777") for p in phases]
for ax, key in zip(axes.flat, plot_keys):
    ax.bar(range(len(phases)), df_phase[key].values, color=colors)
    ax.axhline(gen_summary[key][0], color="k", ls="--", lw=1, label="general")
    ax.set_title(titles[key])
    ax.set_xticks(range(len(phases)))
    ax.set_xticklabels(phases, rotation=40, ha="right", fontsize=8)
    ax.legend(fontsize=8); ax.grid(alpha=.3, axis="y")
fig.suptitle(f"DDPM — métricas físicas por fase (N={n_theta}, K={K})", fontweight="bold")
plt.tight_layout()
plt.savefig("ddpm_physical_metrics_by_phase.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# Distribución de SSIM por fase (boxplot) — diversidad/estabilidad
# ============================================================
fig, ax = plt.subplots(figsize=(11, 5))
data_box = [per_theta["ssim"][(eval_phase == p)] for p in phases]
data_box = [d[~np.isnan(d)] for d in data_box]
bp = ax.boxplot(data_box, labels=phases, patch_artist=True, showfliers=False)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.axhline(gen_summary["ssim"][0], color="k", ls="--", lw=1, label="SSIM general")
ax.set_ylabel("SSIM (media por θ)"); ax.set_title("Distribución de SSIM por fase")
ax.set_xticklabels(phases, rotation=40, ha="right", fontsize=9)
ax.legend(); ax.grid(alpha=.3, axis="y")
plt.tight_layout()
plt.savefig("ddpm_ssim_distribution_by_phase.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# Sanity visual: original vs generada (crop 39) por fase
# ============================================================
n_cols = len(phases)
fig, axes = plt.subplots(2, n_cols, figsize=(2.6 * n_cols, 5.6))
vis_rng = np.random.RandomState(SAMPLE_SEED)
for c, ph in enumerate(phases):
    idx_ph = np.where(eval_phase == ph)[0]
    pick = vis_rng.choice(idx_ph)
    cond_one = torch.from_numpy(eval_cond[pick:pick+1]).to(DEVICE)
    gen40 = fast_sample(model, cond_one, scheduler, n_steps=FAST_STEPS, img_size=IMG_SIZE)
    gen39 = gen40[0, 0, :CROP_TO, :CROP_TO].cpu().numpy()
    real39 = eval_imgs[pick, :, :, 0]
    axes[0, c].imshow(real39, cmap="jet", vmin=-1, vmax=1); axes[0, c].set_title(ph, fontsize=8)
    axes[1, c].imshow(gen39, cmap="jet", vmin=-1, vmax=1)
    for r in range(2):
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Generada (DDPM)", fontsize=10)
fig.suptitle("DDPM — original vs generada por fase", fontweight="bold")
plt.tight_layout()
plt.savefig("ddpm_visual_by_phase.png", dpi=120)
plt.show()
print("Listo. Revisa que las métricas físicas tengan rangos razonables y coherentes por fase.")